# Modelagem estatística da demanda diária por bicicletas

Notebook organizado conforme `README-2.MD`, com entrega em **16/10/2026**. Os blocos executados abaixo cobrem a preparação inicial e a descrição básica. As etapas ainda não implementadas estão identificadas como **Pendentes**; não representam resultados concluídos.

## 1. Objetivo, fonte e variáveis

**Pergunta central:** como temperatura, tipo de dia, sazonalidade mensal e tendência temporal se associam à demanda diária e ajudam a prever datas posteriores ao treino?

Fonte: **Bike Sharing**, UCI Machine Learning Repository, arquivo diário `day.csv`, período de 2011–2012. Cada linha representa um dia; `cnt` é a quantidade de aluguéis, não o número de pessoas ou bicicletas necessárias.

Página oficial: https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset. Referência: Fanaee-T, H. (2013). *Bike Sharing* [Dataset]. https://doi.org/10.24432/C5W894. Licença CC BY 4.0.

| Variável | Significado e uso |
|---|---|
| `dteday` | Data, usada para ordenar registros e separar treino e teste |
| `cnt` | Total de aluguéis; variável dependente |
| `temp` | Temperatura normalizada; preditora quantitativa, sem interpretação direta em °C |
| `workingday` | 1: dia útil; 0: fim de semana ou feriado |
| `mnth` | Mês, de 1 a 12; será tratado como categoria com `C(mnth)` |
| `t_anos` | Tempo desde 01/01/2011 em anos aproximados; criação pendente |
| `casual`, `registered` | Contagens utilizadas para verificar a soma; não entram como preditoras de `cnt` |

`casual + registered = cnt`: incluir essas contagens na previsão do total forneceria a própria resposta ao modelo.


## 2. Leitura e preparação inicial — Pessoa 1

Lemos a base original, reconhecemos as datas e ordenamos os registros. A ordem temporal será necessária na separação dos dados e nos cálculos que consideram dependência entre dias. O arquivo bruto permanece sem alterações.


In [129]:
import pandas as pd
import plotly.express as px

df = pd.read_csv('../dados/brutos/day.csv')

### 2.1 Auditoria

Conferimos ausências, duplicidades, continuidade das datas e consistência das contagens. A continuidade é verificada apenas entre a menor e a maior data registrada. Duas somas em `isna().sum().sum()` contam ausências por coluna e, depois, no conjunto da tabela.


In [122]:
datas_esperadas = pd.date_range(df["dteday"].min(), df["dteday"].max(), freq = "D")

auditoria = pd.Series({
    "registros": len(df),
    "colunas": df.shape[1],
    "valores_ausentes": int(df.isna().sum().sum()),
    "linhas_duplicadas": int(df.duplicated().sum()),
    "datas_duplicadas": int(df["dteday"].duplicated().sum()),
    "dias_sem_registro": len(datas_esperadas.difference(df["dteday"])),
    "totais_inconsistentes": int((df["casual"] + df["registered"] != df["cnt"]).sum()),
    "contagens_negativas": int((df[["casual", "registered", "cnt"]] < 0).sum().sum())
})

auditoria

registros                731
colunas                   16
valores_ausentes           0
linhas_duplicadas          0
datas_duplicadas           0
dias_sem_registro        731
totais_inconsistentes      0
contagens_negativas        0
dtype: int64

A auditoria identifica 731 registros e nenhum problema nas verificações realizadas. Assim, não é necessária imputação nem exclusão automática de registros. Valores extremos, caso investigados posteriormente, não devem ser considerados erros apenas por serem extremos.

### 2.2 Tipos das variáveis utilizadas

As datas devem ter tipo temporal e as contagens e medidas devem ser numéricas. `workingday` e `mnth` usam códigos numéricos, mas representam categorias; o código de um mês não significa que seu efeito cresce linearmente de janeiro a dezembro.


In [123]:
df[[
    "dteday", "cnt", "temp", "workingday", "mnth",
    "casual", "registered"
]].dtypes.to_frame("tipo")


,tipo
dteday,str
cnt,int64
temp,float64
workingday,int64
mnth,int64
casual,int64
registered,int64


### 2.3 Rótulo do tipo de dia

Preservamos `workingday` e criamos apenas o rótulo necessário às tabelas e ao filtro do dashboard. O valor 0 reúne fins de semana e feriados; o valor 1 exclui ambos.


In [124]:
df["tipo_dia"] = df["workingday"].map({
    0: "Fim de semana ou feriado",
    1: "Dia útil"
})

df[["dteday", "workingday", "tipo_dia"]].head()


,dteday,workingday,tipo_dia
0,2011-01-01,0,Fim de semana ou feriado
1,2011-01-02,0,Fim de semana ou feriado
2,2011-01-03,1,Dia útil
3,2011-01-04,1,Dia útil
4,2011-01-05,1,Dia útil


### 2.4 Tendência temporal e exportação — Pendente, Pessoa 1

Criar `t_anos` como os dias decorridos desde 01/01/2011 divididos por 365,25. Essa variável representa uma tendência linear em anos aproximados; não identifica a causa do crescimento.

Depois, salvar a base em `dados/tratados/bicicletas_diario.csv`, sem o índice. A Pessoa 2 deverá usar essa mesma base. Esta seção ainda não contém código de criação ou exportação.


In [126]:
df["dteday"] = pd.to_datetime(df["dteday"])
df["t_anos"] = (df["dteday"] - pd.Timestamp("2011-01-01")).dt.days / 365.25

df[["dteday", "t_anos"]].head()

,dteday,t_anos
0,2011-01-01,0.000000
1,2011-01-02,0.002738
2,2011-01-03,0.005476
3,2011-01-04,0.008214
4,2011-01-05,0.010951


In [128]:
df.to_csv("../dados/tratados/day_tratado.csv", index = False, sep = ';')

## 3. Descrição básica — Pessoa 1

### 3.1 Distribuição da quantidade de aluguéis

O resumo mostra quantidade de observações, média, desvio padrão, mínimo, máximo e quartis. A mediana corresponde ao percentil de 50%; o intervalo entre os quartis de 25% e 75% contém os 50% centrais dos valores.


In [ ]:
resumo_demanda = df["cnt"].describe()

resumo_demanda


count     731.000000
mean     4504.348837
std      1937.211452
min        22.000000
25%      3152.000000
50%      4548.000000
75%      5956.000000
max      8714.000000
Name: cnt, dtype: float64

### 3.2 Comparação por tipo de dia

Comparamos a média de aluguéis por dia e a quantidade de datas observadas. Os totais isolados seriam influenciados pelo maior número de dias úteis. Esta comparação é descritiva: não controla clima, mês ou tendência e não demonstra significância estatística.


In [ ]:
resumo_tipo_dia = (
    df.groupby(["workingday", "tipo_dia"])["cnt"]
    .agg(
        dias_observados="count",
        media_diaria="mean",
        mediana_diaria="median",
        desvio_padrao="std"
    )
    .reset_index()
)

resumo_tipo_dia.round(2)


,workingday,tipo_dia,dias_observados,media_diaria,mediana_diaria,desvio_padrao
0,0,Fim de semana ou feriado,231,4330.17,4459.0,2052.14
1,1,Dia útil,500,4584.82,4582.0,1878.42


Há 500 dias úteis e 231 dias não úteis. As médias são aproximadamente 4.584,82 e 4.330,17 aluguéis por dia, respectivamente: diferença descritiva de cerca de 5,9% em relação aos dias não úteis. Isso não revela o motivo das viagens nem comprova causalidade.

### 3.3 Demanda diária ao longo do tempo — Pendente, Pessoa 1

Construir um gráfico com `dteday` no eixo horizontal e `cnt` no vertical. Seu objetivo é observar evolução e padrões temporais. Uma mudança de nível não identifica sozinha sua causa.




In [130]:
fig_demanda_tempo = px.line(
    df,
    x = "dteday",
    y = "cnt",
    labels = {
        "dteday": "data",
        "cnt": "Aluguéis por dia"
    },
    title = "Demanda diária por bicicletas - 2011-2012",
    template = "plotly_white"
)

fig_demanda_tempo.show()

O gráfico apresenta a evolução diária dos aluguéis em 2011 e 2012. Observa-se maior nível de demanda em boa parte de 2012, além de um padrão de menor utilização no início e no final dos anos e maior utilização nos meses intermediários. Esses padrões motivam considerar tendência temporal e sazonalidade mensal na modelagem. Também existem oscilações diárias e quedas isoladas, cujas causas não podem ser identificadas apenas por essa visualização.

### 3.4 Temperatura × demanda — Pendente, Pessoa 1

Construir uma dispersão de `temp` e `cnt`, com um ponto por dia. Seu objetivo é observar a forma da associação e a dispersão. A temperatura está normalizada e as diferenças de época do ano podem influenciar a relação.

In [136]:
fig_temperatura_demanda = px.scatter(
    df,
    x = "temp",
    y = "cnt",
    hover_data = ["dteday", "workingday"],
    labels = {
        "temp": "Temperatura normalizada",
        "cnt": "Aluguéis por dia",
        "dteday": "Data",
        "tipo_dia": "Tipo de dia"
    },
    title = "Temperatura e demanda diária  - 2011-2012",
    opacity = 0.5,
    template = "plotly_white"
)

fig_temperatura_demanda.show()

A dispersão sugere associação positiva entre temperatura normalizada e demanda diária. Entretanto, dias com temperaturas semelhantes apresentam quantidades bastante diferentes de aluguéis, indicando que a temperatura isoladamente não representa toda a variação observada. Nas temperaturas mais altas, há possível estabilização ou queda da demanda, o que constitui uma limitação a investigar na aproximação linear. A associação visual não demonstra causalidade.


## 4. Separação temporal — Pendente, Pessoa 2

Usar **treino até 30/06/2012**, inclusive, e **teste a partir de 01/07/2012**. Ordenar por data e não embaralhar os registros. O treino serve ao ajuste dos modelos; o teste fica reservado para avaliar previsões de dias posteriores.

A descrição anterior utiliza a série completa como exploração retrospectiva. Os modelos e os testes dos coeficientes abaixo devem utilizar somente o treino; não usar o desempenho no teste para escolher ajustes repetidamente.


## 5. Regressão simples — Pendente, Pessoa 2

Ajustar **`cnt ~ temp`** no treino. Interpretar intercepto e coeficiente da temperatura, considerando sua normalização. Esse modelo será uma comparação simples para o modelo múltiplo.


## 6. Regressão múltipla — Pendente, Pessoa 2

Ajustar **`cnt ~ temp + workingday + C(mnth) + t_anos`** no treino. A temperatura e o tipo de dia são as associações de interesse; mês e tendência representam sazonalidade e evolução temporal.

`C(mnth)` compara os meses com uma categoria de referência, sem impor crescimento linear entre seus códigos. Os coeficientes são associações condicionais às demais variáveis, não efeitos causais.

### 6.1 Diagnóstico básico — Pendente, Pessoa 2

Examinar resíduos versus valores ajustados e autocorrelação dos resíduos. Resíduo é observado menos ajustado: valores positivos indicam subestimação e negativos, superestimação. Procurar padrões e limitações sem excluir registros automaticamente.

Os diagnósticos devem ser produzidos a partir do novo modelo no treino. Não reaproveitar números ou gráficos dos modelos antigos ajustados com os 731 dias.


## 7. Dois testes e intervalos de confiança — Pendente, Pessoa 1

Usar o modelo múltiplo ajustado no treino e erros padrão **HAC com sete defasagens**, com significância de 5% e intervalos de confiança de 95%. Registrar estimativa, intervalo, valor-p e graus de liberdade. Explicar a escolha do uso da distribuição t, caso utilizado.

| Pergunta | H₀ | H₁ |
|---|---|---|
| Há diferença ajustada por tipo de dia? | Coeficiente de `workingday` = 0 | Coeficiente de `workingday` ≠ 0 |
| Há associação ajustada com temperatura? | Coeficiente de `temp` = 0 | Coeficiente de `temp` ≠ 0 |

Os testes surgiram da exploração e devem ser apresentados como evidência exploratória. Valor-p não é a probabilidade de H₀ ser verdadeira; não rejeitar H₀ não comprova igualdade. Discutir magnitude e incerteza, além da significância.

Sete defasagens são um ponto de partida relacionado ao ciclo semanal. HAC ajusta a incerteza diante de possível heterocedasticidade e autocorrelação; não corrige a fórmula nem garante previsões melhores. Recalcular todos os resultados: os testes anteriores não correspondem ao modelo e ao período agora definidos.


## 8. Previsões e desempenho — Pendente, Pessoa 2

Prever julho a dezembro de 2012 com os dois modelos, sem ajustá-los novamente nesses dias. Calcular **MAE** (erro absoluto médio, em aluguéis por dia) e **R²** no teste, identificando o período e a quantidade de observações. R² pode ser negativo quando a soma dos erros quadráticos supera a variabilidade em torno da média do conjunto avaliado.

O clima observado do teste é conhecido retrospectivamente. Uma previsão antecipada exigiria previsão meteorológica ou cenário. Valores ajustados e resíduos do treino não substituem essa avaliação.

Exportar previsões, métricas e tabela dos testes para `resultados/`, com nomes de colunas claros. Construir o gráfico de valores observados e previstos no teste.


## 9. Integração com o dashboard — Pendente, Pessoa 2

Usar Streamlit e Plotly, com filtros de **período** e **tipo de dia**. Incluir três gráficos: demanda ao longo do tempo, temperatura × demanda e observado × previsto no teste. Incluir tabelas por tipo de dia, métricas e testes com intervalos.

Os filtros alteram a exploração descritiva; métricas e testes mantêm seus períodos originais claramente identificados. Não incluir textos que antecipem conclusões ao usuário. Conferir os valores contra o notebook.


## 10. Recomendações e limitações — Interpretação pendente, ambos

Após avaliar os resultados finais, escrever **duas recomendações**. Para cada uma, indicar evidência, ação de planejamento, benefício esperado e limitação. Não inventar economia, aumento de demanda ou capacidade necessária.

Limitações que devem acompanhar as conclusões:

- Associação não demonstra causalidade; os dados não identificam a finalidade das viagens.
- `cnt` é uma contagem; regressão linear pode produzir estimativas negativas.
- O coeficiente de temperatura não representa automaticamente uma mudança de 1 °C.
- Dependência temporal e inadequações da fórmula podem afetar a inferência; HAC não resolve todos esses problemas.
- Previsões antecipadas dependem de informações meteorológicas e da continuidade dos padrões históricos.
- A base não permite redistribuição por estação; aluguéis diários não equivalem à quantidade de bicicletas necessárias.
- Resultados históricos de Washington não se transferem automaticamente para outras cidades ou para o presente.

## 11. Conferência final e defesa — Pendente, ambos

Integrar os trabalhos até quinta-feira; reiniciar o kernel e executar todo o notebook antes da entrega. Manter justificativas e interpretações em Markdown e código sem comentários. Documentar dependências e instruções de execução no README.

Cada integrante deve explicar as variáveis, a exclusão de `casual` e `registered`, a divisão temporal, os coeficientes, os testes, os intervalos, HAC, MAE, R² e a evidência de cada recomendação. A defesa individual representa 80% da nota.
